# Data Prep for SVM

In [1]:
import polars as pl

## Train Test Split

Here we perform the split into train, validation and test data, with a **split** of 70% train, 15% validation and 15% test data. The split is performed **random**. 

In [2]:
SPATIAL_UNIT = "community" # option: census, community, hexa

In [3]:
if SPATIAL_UNIT == "census":
    DATASET = "../data/processed_data/GOLD_HOURLY_DEMAND_CENSUS_TRACT.parquet"
elif SPATIAL_UNIT == "community":
    DATASET = "../data/processed_data/GOLD_HOURLY_DEMAND_COMMUNITY_AREA.parquet"
elif SPATIAL_UNIT == "hexa":
    DATASET = "../data/processed_data/GOLD_HOURLY_DEMAND_HEXAGON.parquet"
else:
    print("Warning: No type of Spatial Data given, Used census tract")
    DATASET = "../data/processed_data/GOLD_HOURLY_DEMAND_CENSUS_TRACT.parquet"

OUTPUT = "../data/train_test_data/"
TARGET_COL = "trip_count"

SEED = 42
RANDOM = True

In [4]:
if(RANDOM == True):
    # split randomly
    df_split = (
        pl.scan_parquet(DATASET)
        .with_row_index("_row_id")
        .with_columns(
            (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
        )
    )

    train = (
        df_split
        .filter(pl.col("_split_bucket") < 70)
        .drop(["_row_id", "_split_bucket"])
    )

    val = (
        df_split
        .filter(
            (pl.col("_split_bucket") >= 70) &
            (pl.col("_split_bucket") < 85)
        )
        .drop(["_row_id", "_split_bucket"])
    )

    test = (
        df_split
        .filter(pl.col("_split_bucket") >= 85)
        .drop(["_row_id", "_split_bucket"])
    )
else :
    # split according to time
    df_split = pl.scan_parquet(DATASET)

    train = df_split.filter(
        pl.col("datetime_hour") < pl.datetime(2025, 9, 1)
    )

    val = df_split.filter(
        (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1)) &
        (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
    )

    test = df_split.filter(
        pl.col("datetime_hour") >= pl.datetime(2026, 1, 1)
    )


total_count = df_split.select(pl.len()).collect().item()
train_count = train.select(pl.len()).collect().item()
val_count = val.select(pl.len()).collect().item()
test_count = test.select(pl.len()).collect().item()

print("Total:", total_count)
print("Train:", train_count, " Share: ", round(train_count / total_count,2))
print("Val:", val_count, " Share: ", round(val_count / total_count,2))
print("Test:", test_count, " Share: ", round(test_count / total_count, 2))

train.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_train.parquet")
val.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_val.parquet")
test.sink_parquet(OUTPUT + "svm_" + SPATIAL_UNIT + "_test.parquet")

Total: 1574265
Train: 1101002  Share:  0.7
Val: 236011  Share:  0.15
Test: 237252  Share:  0.15


In [5]:
type(train)

polars.lazyframe.frame.LazyFrame

In [6]:
df_split.head(10).collect()

_row_id,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,community_area,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,_split_bucket
u32,datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u64
0,2024-07-07 05:00:00,7,7,5,1.2246e-16,-1.0,-0.781831,0.62349,0.965926,0.258819,20.0,75.57,4.0,10.0,0.0,0,0,1,0,0,0,2024-07-07,0,11,38.0,10.0,13.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",55
1,2024-07-10 02:00:00,7,3,2,1.2246e-16,-1.0,0.974928,-0.222521,0.5,0.866025,20.56,88.625,12.0,10.0,0.5101,1,0,0,0,0,0,2024-07-10,0,11,38.0,10.0,13.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",30
2,2024-07-13 01:00:00,7,6,1,1.2246e-16,-1.0,-0.974928,-0.222521,0.258819,0.965926,22.22,75.92,0.0,10.0,0.0,0,1,0,0,0,0,2024-07-13,0,11,38.0,10.0,13.0,4.0,1,84,84.0,84,84,0.08,0.08,0.08,0.08,60.0,60.0,60.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,5.0,5.0,65.0,65.0,65.0,65.0,"""Cash""",67
3,2024-07-13 02:00:00,7,6,2,1.2246e-16,-1.0,-0.974928,-0.222521,0.5,0.866025,22.78,76.01,0.0,10.0,0.0,0,1,0,0,0,0,2024-07-13,0,11,38.0,10.0,13.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",81
4,2024-06-16 21:00:00,6,7,21,0.5,-0.866025,-0.781831,0.62349,-0.707107,0.707107,27.78,50.8,4.0,10.0,0.0,0,0,1,0,0,0,2024-06-16,0,11,38.0,10.0,13.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",82
5,2024-07-10 17:00:00,7,3,17,1.2246e-16,-1.0,0.974928,-0.222521,-0.965926,-0.258819,26.67,50.51,9.0,10.0,0.0,0,0,1,0,0,0,2024-07-10,0,11,38.0,10.0,13.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",13
6,2024-07-10 15:00:00,7,3,15,1.2246e-16,-1.0,0.974928,-0.222521,-0.707107,-0.707107,28.33,47.47,9.0,10.0,0.0,0,0,1,0,0,0,2024-07-10,0,11,38.0,10.0,13.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips""",50
7,2024-06-28 20:00:00,6,5,20,0.5,-0.866025,-0.433884,-0.900969,-0.866025,0.5,23.89,76.19,4.0,10.0,0.0,0,0,1,0,0,0,2024-06-28,0,11,38.0,10.0,13.0,4.0,1,373,373.0,373,373,1.13,1.13,1.13,1.13,6.5,6.5,6.5,6.5,5.0,5.0,5.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,12.0,12.0,12.0,"""Credit Card""",1
8,2024-06-28 11:00:00,6,5,11,0.5,-0.866025,-0.433884,-0.900969,0.258819,-0.965926,21.67,78.54,6.0,10.0,0.76,0,0,1,0,0,0,2024-06-28,0,11,38.0,10.0,13.0,4.0,2,4255,2127.5,1315,2940,17.94,8.97,7.14,10.8,53.0,26.5,20.75,32.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,53.0,26.5,20.75,32.25,"""Prcard""",29


## Feature Selection